# WSJ 2027 - AvdelningskartaKarta över var alla deltagare, ledare, IST och CMT bor, med deltagare ochledare färgade per avdelning (1-53).Avdelningen läses direkt ur Scoutnet, som är facit sedan pushen i juni 2026:question **88168** för deltagare (form 39188) och **107592** för avdelningsledare(form 47115). IST och CMT har ingen avdelning - de jobbar i funktionärsteam -och får egna, enfärgade lager.Koordinaterna är hemadress per person (postnummer-geokodning), med kårensposition som fallback för den som saknar adress i exporten.**Output:** `output/wsj_avdelningar_karta.html`

In [ ]:
import sys, importlib
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
import wsj27_avdelningskarta as ak
importlib.reload(u); importlib.reload(ak)

import pandas as pd

OUTPUT_HTML = '/config/notebooks/wsj27/output/wsj_avdelningar_karta.html'

# Hämta alla bekräftade från Scoutnet och berika med roll + avdelning
raw_data = u.fetch_participants()
df_all, skipped = u.build_participant_dataframe(raw_data)
df = ak.enrich_with_avdelning(df_all, raw_data)

# Hemadress per person (postnummer via GeoNames), kårens medlemsmedian som fallback
ak.assign_map_coordinates(df)

print(f'\nTotalt på kartan: {len(df)} personer')

In [ ]:
# Statistik per avdelning - antal deltagare, ledare och geografisk spridning
ak.print_avdelning_summary(df)

In [ ]:
# Generera kartan
ak.generate_avdelning_map_html(df, output_path=OUTPUT_HTML)

In [ ]:
# IST och CMT - de som inte har någon avdelning
df_ovriga = df[df['roll'].isin(['ist', 'cmt'])]
print(f'=== IST och CMT: {len(df_ovriga)} personer utan avdelning ===\n')

print('Per roll och resetyp:')
print(df_ovriga.groupby(['roll', 'travel']).size().to_string())

print('\nTopp 15 kårer:')
for kar, n in df_ovriga['kar'].value_counts().head(15).items():
    print(f'  {n:>3}  {kar or "(ingen kår)"}')

print('\nPer region:')
for region, n in df_ovriga['region'].value_counts().items():
    print(f'  {n:>3}  {region or "(okänd)"}')